# 02 — Metrics

**What do these numbers mean, and what can they not tell you?**

Read this before any experiment. Several of the metrics we rely on answer a
narrower question than their name suggests, and the difference decides
whether a result is real.

| section | |
|---|---|
| 1 | the catalogue — every metric, by the question it answers, and its blind spot |
| 2 | the SLAs — each target, why that number, which metric proves it |
| 3 | percentiles — why histograms, and when a percentile is not trustworthy |
| 4 | component checks — each component judged by its own metrics |
| 5 | watch them move — own seeded load, with the real Grafana panels |

Independent: drives its own load, needs nothing from 01. Just `make ssh`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, time, random
sys.path.insert(0, '..')

from utils import Cluster, Report, filler
from prom import (Prom, Grafana, CATALOGUE, LATENCY_HISTOGRAMS,
                  panel_id, panel_queries, metric_name, GRAFANA_URL)
from slos import SLOS, table_rows

cluster = Cluster()
prom    = Prom()
grafana = Grafana(password=os.environ.get('GRAFANA_PASSWORD', ''))
report  = Report('Dodona 02 — metrics', 'what the numbers mean')

report.objective(
    'What does each metric mean, what does good look like, and what can it '
    'NOT tell you?',
    falsified_by='a metric in the catalogue that no longer exists, or a trap '
                 'that no longer reproduces')


## 0. Generate the traffic this notebook reads

First, so every section below reads data from **this run** rather than
whatever the cluster happened to do earlier. 8 short requests and 4 long
**unique** ones — unique matters, because a repeated prompt is served from
decode's own cache and nothing crosses between pods.

Then one scrape interval, so Prometheus has it.


In [ ]:
report.heading('0. Traffic for this run')
random.seed(2)

sent, failed = 0, {}
for i in range(8):
    r = cluster.chat('Reply with the single word: ok', max_tokens=8); sent += 1
    if '_error' in r: failed[r['_error']] = failed.get(r['_error'], 0) + 1
for i in range(4):
    r = cluster.chat(f'[{i}-{random.random()}] ' + filler(4000) + '\n\nReply with: ok',
                     max_tokens=8, timeout=180); sent += 1
    if '_error' in r: failed[r['_error']] = failed.get(r['_error'], 0) + 1

time.sleep(20)   # one scrape interval at 15s
report.table(['', 'count'], [['requests sent', sent],
                            *[[f'refused: {k}', n] for k, n in failed.items()]],
             note='Refusals here are gateway decisions, not GPU limits.')
report.verdict(sent - sum(failed.values()) > 0, 'some traffic completed')
report.conclusion(f'{sent} requests, {sum(failed.values())} refused. Every number '
                  'below covers a 15-minute window that includes this traffic.')


## 1. The catalogue

Grouped by the **question**, not the metric name. A name tells you nothing
about whether its value is good.

The last column is the one that matters: **what this metric cannot tell you.**
Twenty of the twenty-three carry one.


In [ ]:
# metric name first, then the value, then what it means — a reader needs
# to know WHICH series they would go and look at.
for question, entries in CATALOGUE.items():
    report.heading(question)
    rows = []
    for label, expr, means, blind in entries:
        got = prom.query(expr)
        value = f'{got[0][1]:,.4g}' if got else ('—' if not prom.up() else 'no data')
        rows.append([metric_name(expr), value, label, means])
    report.table(['metric', 'now', 'answers', 'means'], rows)
    for label, expr, _, blind in entries:
        if blind:
            report.text(f'{metric_name(expr)} — CANNOT TELL YOU: {blind}')


## 2. The SLAs

A measurement is a number. A measurement against a promise is a result.

These live in `benchmarking/slos.py`, so every notebook checks the same ones
and nothing drifts between documents.


In [ ]:
report.heading('2. The SLAs')
rows = []
for key, s in SLOS.items():
    rows.append([s.label, f'{s.target:g}{s.unit}', s.why])
report.table(['promise', 'target', 'why that number'], rows)
report.text('Concurrency is deliberately absent: it is derived in 03-capacity '
            'from a measured saturation curve. A concurrency target picked '
            'before measuring is a wish.')
report.conclusion(
    'Each has a metric that proves it: TTFT and turn from the latency '
    'histograms, preemptions from vllm:num_preemptions_total, shed rate from '
    'orch_shed_total over orch_requests_total.')


## 3. Percentiles

vLLM publishes **histograms**. The percentile comes from the buckets:

```promql
histogram_quantile(0.99, sum by (le) (rate(vllm:time_to_first_token_seconds_bucket[5m])))
```

Sorting ten samples in Python gives a "p99" that is the maximum of ten and
swings wildly between runs. This notebook did that until recently.

**What each hides from the others:** p50 is the typical experience and says
nothing about the tail. p99 is the tail and says nothing about how common it
is. A p50 of 0.3s with a p99 of 8s is a system that works until it doesn't.


In [ ]:
report.heading('3. Percentiles')
rows, series, quantile_key = [], {}, {}
for name in ('ttft', 'tpot', 'e2e', 'queue', 'prefill', 'decode'):
    q = prom.quantiles(name, qs=(0.5, 0.9, 0.99))
    stats = prom.histogram_stats(name)
    if not q and not stats:
        continue
    # The mean from _sum/_count is exact. The quantiles are interpolated
    # inside a bucket, so they are only as good as the bucket layout.
    # A missing quantile is "no samples in the window", not zero latency.
    cell = lambda x: f'{x:.4f}' if x is not None else '—'
    rows.append([name, f"{stats.get('count', 0):,.0f}",
                 f"{stats.get('mean', 0):.4f}",
                 cell(q.get(0.5)), cell(q.get(0.9)), cell(q.get(0.99)),
                 f'{q[0.99]/q[0.5]:.1f}x' if q.get(0.5) else '—'])
    if q:
        series[name] = [q.get(0.5,0), q.get(0.9,0), q.get(0.99,0)]
        quantile_key.setdefault(tuple(round(x, 6) for x in series[name]), []).append(name)

if rows:
    report.table(['metric', 'n', 'mean s', 'p50 s', 'p90 s', 'p99 s', 'p99/p50'], rows,
                 note='mean is exact (_sum/_count). The percentiles are '
                      'interpolated inside a bucket.')
    report.bar('Percentiles by metric', ['p50', 'p90', 'p99'], series,
               ylabel='seconds (log)', log=True,
               note='tpot is per OUTPUT TOKEN, orders of magnitude below e2e')

    # Two metrics reporting the same quantile to six decimals are not equal;
    # they are landing in the same bucket. The means say so.
    collisions = {k: v for k, v in quantile_key.items() if len(v) > 1}
    report.verdict(not collisions,
                   'no two metrics report identical quantiles')
    if collisions:
        for _, names in collisions.items():
            means = ', '.join(
                f"{n} {prom.histogram_stats(n).get('mean', 0):.4f}s" for n in names)
            report.text(f'{" and ".join(names)} report the SAME percentiles — '
                        f'a bucket artefact, not equality. True means: {means}')
        report.conclusion(
            'Identical percentiles across different metrics mean the data is '
            'concentrated and histogram_quantile is returning a bucket edge. '
            'Quote the mean at this sample size, or generate more load — the '
            'percentiles become meaningful once the buckets are populated.')
    else:
        report.conclusion(
            'Means and percentiles are consistent, so the buckets are populated '
            'enough for the percentiles to carry information.')
else:
    report.text('No histogram data yet — section 5 generates some.')


## 4. Component checks

Each component, its own metrics, and a verdict. First principles: what is
this component supposed to do, which number says whether it did it.

The *cannot tell you* column in section 1 is what keeps these honest — several
of these metrics answer a narrower question than their name suggests.


In [ ]:
report.heading('4. Component checks')
v = lambda expr: (prom.query(expr)[0][1] if prom.query(expr) else 0.0)

for group, checks in COMPONENT_CHECKS.items():
    report.heading(group)
    rows = []
    for label, expr, meaning in checks:
        got = prom.query(expr)
        value = f'{got[0][1]:,.4g}' if got else 'no data'
        rows.append([metric_name(expr), value, meaning])
    report.table(['metric', 'value', 'what it means'], rows)


### 4.1 The KV connector

It exists to let decode load the prompt's KV from prefill instead of
recomputing it. Three numbers, in the order the request passes through them:
**asked → found → arrived.**

`external_prefix_cache_hits_total` records that the *lookup* matched. The
pull happens after, and can fail. So `found` and `arrived` agreeing is what
says the connector is doing its job end to end.


In [ ]:
report.heading('4.1 The KV connector')
asked    = v('sum(vllm:external_prefix_cache_queries_total)')
found    = v('sum(vllm:external_prefix_cache_hits_total)')
arrived  = v('sum(vllm:prompt_tokens_by_source_total{source="external_kv_transfer"})')
computed = v('sum(vllm:prompt_tokens_by_source_total{source="local_compute"})')
cached   = v('sum(vllm:prompt_tokens_by_source_total{source="local_cache_hit"})')

report.table(['stage', 'tokens', ''], [
    ['asked',   f'{asked:,.0f}',   'decode looked for blocks on the producer'],
    ['found',   f'{found:,.0f}',   'the lookup matched'],
    ['arrived', f'{arrived:,.0f}', 'KV that came over the wire'],
])
report.verdict(arrived > 0, 'the connector has delivered KV')
report.verdict(abs(found - arrived) < max(found, 1) * 0.1,
               'found and arrived agree — lookups are completing as transfers')

report.bar('Where decode gets the prompt KV it needs',
           ['computed here', 'its own prefix cache', 'from the connector'],
           {'prompt tokens': [computed, cached, arrived]}, ylabel='tokens',
           note='the first two are local to decode; only the third crossed between pods')

total = computed + cached + arrived
report.conclusion(
    f'Of {total:,.0f} prompt tokens decode needed, {arrived:,.0f} ({arrived/max(total,1):.1%}) '
    f'arrived as KV from prefill, {cached:,.0f} came from its own prefix cache, and '
    f'{computed:,.0f} it computed. The connector is working.'
    if arrived > 0 else
    f'None of the {total:,.0f} prompt tokens came from the connector. '
    + (f'It was asked {asked:,.0f} times and found {found:,.0f}, so the lookup '
       'works and the transfer does not — check `make kv` for transfer failures.'
       if found > 0 else 'It found nothing, so there is nothing to transfer.'))


### 4.2 Prefill and decode are doing different jobs

The split is only real if the two pools behave differently. Prefill's leg is
sent with `max_tokens=1`: enough to build the prompt's KV, not to answer.
So prefill should emit **one token per request** and decode should do the
generating.


In [ ]:
report.heading('4.2 Prefill and decode')
pre_gen = v('sum(vllm:generation_tokens_total{pod=~"vllm-prefill.*"})')
dec_gen = v('sum(vllm:generation_tokens_total{pod=~"vllm-decode.*"})')
served  = v('sum(orch_requests_total)')
per_req = pre_gen / served if served else 0

report.table(['', 'tokens', ''], [
    ['prefill generated', f'{pre_gen:,.0f}', f'{per_req:.2f} per request'],
    ['decode generated',  f'{dec_gen:,.0f}', 'the actual answers'],
])
report.verdict(served == 0 or per_req < 1.5,
               f'prefill emits {per_req:.2f} tokens per request (max_tokens=1 means ~1.0)')
report.conclusion(
    f'Prefill emitted {per_req:.2f} tokens per request against decode\'s '
    f'{dec_gen:,.0f} total — the two pools are doing different work, which is '
    'what disaggregation means.'
    if per_req < 1.5 else
    f'Prefill emitted {per_req:.2f} tokens per request. A max_tokens=1 leg cannot '
    'do that, so both legs are sending the full request.')


### 4.3 The slice, not the card

`kv_cache_usage_perc` is a ratio of **one pod's own** KV. On this cluster each
pod has a 12 GiB HAMi slice, not the whole card — so the absolute numbers
differ by about 15x and only one of them is the right denominator.


In [ ]:
report.heading('4.3 Slice capacity')
report.table(['', 'KV available', 'full-length sequences @16k'], [
    ['the whole H100', '~64 GiB', '~73.5'],
    ['one 12 GiB HAMi slice', '4.10 GiB', '4.7'],
], note='CUDA-graph capture scales with available memory, so the whole-card '
        'figure cannot simply be divided down — it is measured per slice.')
kv_now = v('max(vllm:kv_cache_usage_perc)')
report.conclusion(
    f'KV usage is {kv_now:.1%} of a 4.10 GiB slice, which is {kv_now*4.10:.2f} GiB — '
    f'not {kv_now:.1%} of the card. The pool is 4.7 concurrent sequences, and that '
    'is the number every concurrency claim divides by.')


## 5. The same numbers, as Grafana draws them

Rendered server-side, not redrawn — a screenshot of the panel the team
already watches needs no explanation of how it was derived.

Needs `GRAFANA_PASSWORD` and the renderer deployment; falls back to links.


In [ ]:
from pathlib import Path
shots = Path('screenshots'); shots.mkdir(exist_ok=True)
wanted = [('dodona-vllm', 'vllm', 'GPU KV cache'), ('dodona-vllm', 'vllm', 'TTFT'),
          ('dodona-mooncake', 'mooncake', 'DELIVERED'),
          ('dodona-mooncake', 'mooncake', 'Claimed vs delivered'),
          ('dodona-gateway', 'gateway', 'shed')]
saved = []
for uid, dash, title in wanted:
    pid = panel_id(dash, title)
    if pid is None: continue
    out = grafana.render(uid, pid, shots / f'{dash}-{title[:18].lower().replace(" ","-")}.png')
    if out: saved.append(out)

if saved:
    report.verdict(True, f'rendered {len(saved)} Grafana panels')
    from IPython.display import Image, display
    for s in saved: display(Image(filename=str(s)))
else:
    report.text('No panels rendered — set GRAFANA_PASSWORD (printed by '
                'make observability) and check renderer.yaml is deployed.')
    report.links([(d, f'{GRAFANA_URL}/d/{u}?from=now-30m&to=now')
                  for u, d in dict.fromkeys((u, d) for u, d, _ in wanted)])


## Save


In [ ]:
path  = report.save('02-metrics')
fails = report.failures()
print('wrote', path)
print()
if fails:
    for f in fails: print('  open:', f)
else:
    print('Every metric accounted for. Next: 03-capacity')
